# Network Intrusion Detection: End-to-End Multiclass Pipeline

This notebook trains a 5-fold Stratified cross-validation model for the Network Intrusion Detection Challenge:
1. **Preprocesses data** and handles missing values/infinities.
2. **Engineers domain-specific network traffic flow features** (port ranges, packet ratios, TCP flag balances, window sizes).
3. **Applies balanced sample weights** to handle the 94.5% vs 0.87% class imbalance.
4. **Trains a 5-Fold Stratified XGBoost ensemble** optimizing Macro F1 Score.
5. **Generates predictions** and formats `submission.csv`.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb

# Set file paths
DATA_DIR = Path("../data")
train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
print(f"Loaded train: {train_df.shape}, test: {test_df.shape}")

In [ ]:
CLASSES = ["Benign", "Availability", "Reconnaissance", "Spoofing"]
LABEL_TO_IDX = {l: i for i, l in enumerate(CLASSES)}
IDX_TO_LABEL = {i: l for i, l in enumerate(CLASSES)}

# Feature engineering function
def engineer_features(df):
    df = df.copy()
    eps = 1e-5
    
    # Port categories
    df["orig_port_well_known"] = (df["id.orig_p"] < 1024).astype(float)
    df["orig_port_registered"] = ((df["id.orig_p"] >= 1024) & (df["id.orig_p"] < 49152)).astype(float)
    df["orig_port_dynamic"] = (df["id.orig_p"] >= 49152).astype(float)
    df["resp_port_well_known"] = (df["id.resp_p"] < 1024).astype(float)
    df["resp_port_registered"] = ((df["id.resp_p"] >= 1024) & (df["id.resp_p"] < 49152)).astype(float)
    df["resp_port_dynamic"] = (df["id.resp_p"] >= 49152).astype(float)
    
    # Packet & byte ratios
    df["pkts_tot"] = df["fwd_pkts_tot"] + df["bwd_pkts_tot"]
    df["pkts_ratio"] = (df["fwd_pkts_tot"] + eps) / (df["bwd_pkts_tot"] + eps)
    df["payload_tot"] = df["fwd_pkts_payload.tot"] + df["bwd_pkts_payload.tot"]
    df["payload_ratio"] = (df["fwd_pkts_payload.tot"] + eps) / (df["bwd_pkts_payload.tot"] + eps)
    
    # Headers & flags
    df["header_size_tot"] = df["fwd_header_size_tot"] + df["bwd_header_size_tot"]
    df["header_to_payload_ratio"] = (df["header_size_tot"] + eps) / (df["payload_tot"] + eps)
    df["syn_ack_ratio"] = (df["flow_SYN_flag_count"] + eps) / (df["flow_ACK_flag_count"] + eps)
    df["window_diff"] = df["fwd_init_window_size"] - df["bwd_init_window_size"]
    
    # Log transforms
    for col in ["flow_duration", "flow_pkts_per_sec", "payload_bytes_per_second", "payload_tot"]:
        if col in df.columns:
            df[f"{col}_log1p"] = np.log1p(np.clip(df[col], 0, None))
            
    num_cols = df.select_dtypes(include=[np.number]).columns
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
    return df

train_feats = engineer_features(train_df)
test_feats = engineer_features(test_df)

# Categorical encoding
for c in ["proto", "service"]:
    train_feats[f"{c}_enc"] = train_feats[c].astype("category").cat.codes
    test_feats[f"{c}_enc"] = test_feats[c].astype("category").cat.codes

feature_cols = [c for c in train_feats.columns if c not in ["id", "Target", "proto", "service"]]
X = train_feats[feature_cols].copy()
y = train_feats["Target"].map(LABEL_TO_IDX).values
X_test = test_feats[feature_cols].copy()
print(f"Feature matrix prepared with {len(feature_cols)} features.")

In [ ]:
# Balanced class weighting for Macro-F1
classes = np.unique(y)
weights = compute_class_weight("balanced", classes=classes, y=y)
weight_dict = dict(zip(classes, weights))
sample_weights = np.array([weight_dict[c] for c in y])

# 5-Fold Stratified Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train_df), len(CLASSES)))
test_probs = np.zeros((len(test_df), len(CLASSES)))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"--- Fold {fold + 1}/5 ---")
    model = xgb.XGBClassifier(
        n_estimators=250, learning_rate=0.08, max_depth=6,
        tree_method="hist", random_state=42 + fold, n_jobs=-1
    )
    model.fit(X.iloc[train_idx], y[train_idx], sample_weight=sample_weights[train_idx])
    val_pred = model.predict(X.iloc[val_idx])
    print(f"Fold {fold + 1} Macro F1: {f1_score(y[val_idx], val_pred, average='macro'):.4f}")
    test_probs += model.predict_proba(X_test) / 5

# Format and export submission.csv
sub_preds = np.argmax(test_probs, axis=1)
sub_df = pd.DataFrame({
    "id": test_df["id"],
    "Target": [IDX_TO_LABEL[i] for i in sub_preds]
})
sub_df.to_csv("../submissions/submission.csv", index=False)
print("Submission generated successfully!")
print(sub_df["Target"].value_counts())